# Cloudflare AI Search

This notebook uses the official Cloudflare Python SDK to create and populate an AI Search instance, then queries it through the LangChain `CloudflareAISearchRetriever` integration.

## Setup

Install the Cloudflare Python SDK and the LangChain Cloudflare integration.

In [ ]:
%pip install -qU cloudflare langchain-cloudflare python-dotenv

### Credentials

Create a Cloudflare API token with **AI Search:Edit** and **AI Search:Run** permissions. Set the following values in your environment or `.env` file:

- `CLOUDFLARE_ACCOUNT_ID` and `CLOUDFLARE_API_TOKEN`, matching the official Cloudflare SDK conventions; or
- `CF_ACCOUNT_ID` and `CF_AI_SEARCH_API_TOKEN`, matching the `langchain-cloudflare` conventions.

The optional `CF_AI_SEARCH_NAMESPACE` value defaults to `default`.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(".env")

account_id = os.getenv("CLOUDFLARE_ACCOUNT_ID") or os.getenv("CF_ACCOUNT_ID")
api_token = (
    os.getenv("CF_AI_SEARCH_API_TOKEN")
    or os.getenv("CLOUDFLARE_API_TOKEN")
    or os.getenv("CF_API_TOKEN")
)
namespace = os.getenv("CF_AI_SEARCH_NAMESPACE", "default")

if not account_id or not api_token:
    raise ValueError("Set a Cloudflare account ID and AI Search API token")

## Initialization

Use a unique instance name and query marker so this run cannot collide with another notebook session.

In [ ]:
import time
import uuid

from cloudflare import Cloudflare
from langchain_cloudflare import CloudflareAISearchRetriever

cloudflare = Cloudflare(api_token=api_token)
instance_name = f"langchain-ai-search-{uuid.uuid4().hex[:8]}"
query_marker = f"langchain-ai-search-notebook-{uuid.uuid4().hex}"

instance_name

## Create an AI Search instance

The `default` namespace exists automatically. Because this instance does not specify another data source, AI Search provisions built-in storage for direct file uploads.

In [ ]:
instance = cloudflare.aisearch.namespaces.instances.create(
    name=namespace,
    account_id=account_id,
    id=instance_name,
)

instance.id

## Upload content

Upload a Markdown document to the instance's built-in storage. The upload returns without waiting for indexing, and the next cell polls the item status with a bounded timeout.

In [ ]:
content = "\n".join(
    [
        "# LangChain Cloudflare AI Search notebook",
        "",
        f"{query_marker} validates AI Search retrieval through LangChain.",
        "Cloudflare AI Search indexes uploaded files for natural language search.",
    ]
)

item = cloudflare.aisearch.namespaces.instances.items.upload(
    id=instance_name,
    account_id=account_id,
    name=namespace,
    file={
        "file": ("langchain-guide.md", content.encode(), "text/markdown"),
        "wait_for_completion": False,
    },
)

item.status

In [ ]:
deadline = time.monotonic() + 240

while item.status in {"queued", "running"} and time.monotonic() < deadline:
    time.sleep(3)
    item = cloudflare.aisearch.namespaces.instances.items.get(
        item.id,
        id=instance_name,
        account_id=account_id,
        name=namespace,
    )

if item.status != "completed":
    raise RuntimeError(f"Indexing did not complete: {item.status}: {item.error}")

item

## Search with the Cloudflare SDK

First, issue a raw AI Search query through the official SDK. The response contains the matching indexed chunks.

In [ ]:
results = cloudflare.aisearch.namespaces.instances.search(
    id=instance_name,
    account_id=account_id,
    name=namespace,
    query=query_marker,
)

[(chunk.text, chunk.score) for chunk in results.chunks]

## Retrieve with LangChain

`CloudflareAISearchRetriever` converts AI Search chunks into LangChain `Document` objects. It can be used directly in chains, agents, and retriever tools.

In [ ]:
retriever = CloudflareAISearchRetriever(
    account_id=account_id,
    api_token=api_token,
    namespace=namespace,
    instance_name=instance_name,
    k=3,
    rewrite_query=False,
    reranking=False,
)

documents = retriever.invoke(query_marker)
[(document.page_content, document.metadata) for document in documents]

## Python Workers

The Cloudflare Python SDK can run in Python Workers through its asynchronous client:

```python
from cloudflare import AsyncCloudflare

client = AsyncCloudflare(api_token=env.CLOUDFLARE_API_TOKEN)
results = await client.aisearch.namespaces.instances.search(
    id="my-instance",
    account_id=env.CLOUDFLARE_ACCOUNT_ID,
    name="default",
    query="How does AI Search work?",
)
```

For retrieval inside a Worker, a dedicated AI Search binding avoids storing an API token and making an outbound REST request:

```python
retriever = CloudflareAISearchRetriever(binding=env.MY_SEARCH)
documents = await retriever.ainvoke("How does AI Search work?")
```

## Cleanup

Delete only the item and instance created by this notebook run.

In [ ]:
cloudflare.aisearch.namespaces.instances.items.delete(
    item.id,
    id=instance_name,
    account_id=account_id,
    name=namespace,
)
cloudflare.aisearch.namespaces.instances.delete(
    instance_name,
    account_id=account_id,
    name=namespace,
)

## API reference

- [Cloudflare AI Search Python SDK guide](https://developers.cloudflare.com/ai-search/get-started/python/)
- [Cloudflare Python SDK](https://developers.cloudflare.com/api/python/)
- [Cloudflare AI Search Workers binding](https://developers.cloudflare.com/ai-search/api/instances/workers-binding/)
- [LangChain retriever concepts](https://python.langchain.com/docs/concepts/retrievers/)